# Mushroom Confusion Diagnosis — Run 5 + Grad-CAM


## Objective

Run 3/4 showed mushroom stuck at ~78-85% val_accuracy with a large train/val gap (~99.9% vs
~84%) — textbook overfitting signature. Run 5 tested that hypothesis directly: add regularization
(stronger augmentation, weight decay, label smoothing) and see if the gap closes. It didn't. This
notebook documents Run 5, the corrected comparison against Run 4, and a Grad-CAM investigation
into *why* — which points at a different diagnosis than overfitting.

Kept separate from `07_training_run_log.ipynb` since this is a focused investigation with
generated diagnostic images, not a routine run log entry.


## What changed for Run 5

Three cheap, standard regularizers, all newly config-driven (both `build_loss` and
`build_optimizer` already accepted `**kwargs`, so no new plumbing needed beyond reading them
from `training_config` in `scripts/train_baseline.py`):

- `augmentation_preset: light → medium` in `configs/mushroom.yaml` (rotation + stronger color
  jitter, on top of the existing flip)
- `weight_decay: 0.05` (up from AdamW's implicit 0.01 default)
- `label_smoothing: 0.1` (softens target confidence in the cross-entropy loss)

Also fixed an unrelated infrastructure problem discovered during Run 4: `scripts/train_truba_cpu.sbatch`
now requests `--exclusive` — Run 4 shared its node with another job (`CPUAlloc=61` when we only
asked for 56) and epoch times swung between 2900-4760s as a result. Run 5 confirms the fix: epoch
times are flat at ~950s throughout (see the chart below).


![Mushroom Run 5 metrics](assets/mushroom_run5_metrics.png)


## Run 4 vs Run 5 — corrected comparison

The quick read from the W&B charts alone made Run 5 look roughly equivalent to Run 4. Pulling the
exact numbers from both runs' best epoch (by `val_macro_f1`) tells a clearer story:

| | Run 4 (light aug, no weight decay) | Run 5 (medium aug + weight_decay + label_smoothing) |
|---|---|---|
| Best epoch | 18 | 24 |
| val_accuracy | **84.80%** | 83.27% |
| val_macro_f1 | **0.8126** | 0.7911 |
| train_accuracy (same epoch) | 99.93% | 99.95% |
| train/val accuracy gap | ~15.1 pts | ~16.7 pts |

**Regularization made things slightly worse, not better, on every metric that matters, and the
train/val gap didn't shrink either** — train accuracy is still ~99.9%+ regardless of augmentation
strength or weight decay. `val_loss`'s absolute value did go up in Run 5 (1.5 vs 0.88 in Run 4),
but that's an artifact of label smoothing changing the loss function's floor (it never lets loss
approach zero, even for perfect predictions) — not a real regression, and not comparable across
the two runs directly. Accuracy and macro-F1 are what's comparable, and both point the same way.

**This is the actual finding that matters**: if regularization aimed at "the model is
memorizing the training set" doesn't move the needle at all, the working hypothesis
("overfitting due to insufficient regularization") is probably wrong, or at least incomplete.


## The confused-pairs pattern

Run 5's top-10 confused pairs (from `outputs/reports/mushroom_resnet50_eval_report.json`):

| true → predicted | count |
|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 64 |
| Fomes fomentarius → Fomitopsis betulina | 31 |
| Amanita muscaria → Amanita persicina | 30 |
| Evernia prunastri → Evernia mesomorpha | 25 |
| Fomitopsis pinicola → Fomes fomentarius | 24 |
| Fomes fomentarius → Ganoderma applanatum | 21 |
| Parmelia sulcata → Hypogymnia physodes | 21 |
| Pleurotus pulmonarius → Pleurotus ostreatus | 21 |
| Xanthoria parietina → Vulpicida pinastri | 21 |
| Leccinum scabrum → Leccinum aurantiacum | 18 |

**Every single one of these pairs is a same-genus confusion**: *Fomitopsis* with *Fomitopsis*,
*Amanita* with *Amanita*, *Evernia* with *Evernia*, *Pleurotus* with *Pleurotus*, *Leccinum* with
*Leccinum* — the two *Fomitopsis pinicola* rows and the *Fomes fomentarius* → *Ganoderma
applanatum* row are all bracket/polypore fungi that look alike even to the model apparently
across genus lines too. This is not noise — it's a consistent, taxonomically coherent error
pattern, which is a very different signal than "the model overfit the training set" would
produce (that would look like more random, spread-out confusion, not concentrated on
closely-related species pairs).

This reframes the question from *"how do we stop the model from memorizing?"* to *"can a
224×224 ResNet-50 tell these specific look-alike species apart at all, and if not, why?"*
— which is exactly what Grad-CAM can help answer: is the model looking at the right part of the
image and still failing (a genuine fine-grained discrimination limit), or is it looking
somewhere irrelevant (a fixable data/pipeline issue)?


## Grad-CAM: where is the model looking when it gets these wrong?

Implemented in `src/explainability/gradcam.py` — a standard Grad-CAM (Selvaraju et al., 2017):
hook the last conv block (`model.backbone.layer4`), backprop the predicted class's score, weight
the activation channels by their gradients, and overlay the resulting heatmap on the input image.

For each of the top 5 confused pairs, loaded the Run 5 best checkpoint, ran inference over the
full validation set, found actual misclassified examples (`true_label == A, predicted_label ==
B`), and ran Grad-CAM on 3 random examples per pair. Red/yellow = where the model's prediction is
most sensitive to; blue = ignored.


![Grad-CAM: Fomitopsis pinicola misclassified as Fomitopsis mounceae](assets/gradcam_Fomitopsis_pinicola_Fomitopsis_mounceae.png)


![Grad-CAM: Fomes fomentarius misclassified as Fomitopsis betulina](assets/gradcam_Fomes_fomentarius_Fomitopsis_betulina.png)


![Grad-CAM: Amanita muscaria misclassified as Amanita persicina](assets/gradcam_Amanita_muscaria_Amanita_persicina.png)


![Grad-CAM: Evernia prunastri misclassified as Evernia mesomorpha](assets/gradcam_Evernia_prunastri_Evernia_mesomorpha.png)


![Grad-CAM: Fomitopsis pinicola misclassified as Fomes fomentarius](assets/gradcam_Fomitopsis_pinicola_Fomes_fomentarius.png)


## Reading the Grad-CAM results

**In all 15 examples across all 5 pairs, the model's attention sits squarely on the fungus or
lichen itself** — never on the bark, snow, moss, hand, or background. There is no example of the
model "cheating" by keying off an irrelevant cue. That rules out the most easily-fixable
explanation (a background/shortcut-learning artifact) and supports the taxonomic-confusion
reading: **the model is looking at exactly the right structures and still can't separate these
species.**

A secondary pattern in the confidence scores is worth noting: the *Fomitopsis pinicola ↔
mounceae* and *Evernia prunastri ↔ mesomorpha* pairs get consistently high-confidence wrong
predictions (0.76-0.96) with attention covering the whole fruiting body/thallus — the model isn't
uncertain, it's confidently wrong, which is consistent with these species being genuinely
close to indistinguishable at this resolution/scale. The *Fomes fomentarius ↔ Fomitopsis
betulina* pair instead shows lower, more varied confidence (0.41-0.96) and attention sometimes
concentrated on a smaller sub-region rather than the whole structure — a messier, more
borderline case, possibly a harder pair even for the correct-answer cases, or images where the
diagnostic surface (pore structure, which isn't very visible in bracket-fungus photos taken from
this angle) simply isn't captured by the photo.

**What this does and doesn't tell us:**
- Does *not* support: more/heavier regularization (Run 5 already tested this, no effect), or
  suspecting a background/data-leakage artifact (Grad-CAM shows none).
- Does support treating this as a genuine fine-grained visual discrimination problem, where the
  next things worth testing (in order) are things that affect how much visual detail the model
  can actually see and use, not things that fight overfitting:
  1. **Higher input resolution** (384×384, already benchmarked as an option in Sprint 3) — 224px
     may be discarding the fine surface/texture detail that separates e.g. *Fomitopsis pinicola*
     from *mounceae*.
  2. **A different architecture** (EfficientNet-B3, per the Sprint 4 research note) — compare
     whether it makes the *same* genus-level mistakes. If yes, this is a dataset/resolution
     ceiling, not a ResNet-50-specific weakness. If no, architecture matters more than expected
     here.
  3. Only after 1-2: consider whether these specific confused species need targeted extra data,
     rather than assuming more data across the board would help evenly.
